# LAB : Agents avec Langchain

**SMA et IAD - Master SDIA**

**Enseignant :** Prof. RETAL SARA

**Étudiant :** HYNDI ELMEHDI  
**Date :** 8 Juin 2026

---

## 1. Création d’un nouveau agent sans System Message

Dans cette section, nous créons un agent ReAct de base sans lui donner d'instructions particulières (System Message) et lui posons une question absurde.

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Initialiser le modèle Ollama
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

agent = create_agent(model=model)
question = HumanMessage(content="Quelle est la capitale de la lune ?")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

La question classique !

Il n'y a pas de capitale sur la Lune, car elle n'est pas habitée par des êtres humains ou d'autres espèces. La Lune est un satellite naturel de la Terre et n'a pas de gouvernement ou de population permanente.

Cependant, il y a eu plusieurs missions spatiales qui ont envoyé des robots et des astronautes sur la surface de la Lune. Parmi ceux-ci, on peut citer les missions Apollo américaines, qui ont envoyé des astronautes sur la Lune en 1969-1972.

Si vous cherchez à connaître le nom d'une station ou d'un site spécifique sur la Lune, je peux vous aider !


## 2. Agent avec System Message

Le system message définit le comportement global du modèle. Ici, nous lui demandons de se comporter comme un auteur de science-fiction.

In [12]:
system_prompt = "Vous êtes un auteur de science-fiction ; créez une capitale à la demande des utilisateurs."
scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)
response = scifi_agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

La question classique !

Malheureusement, il n'y a pas de capitale sur la Lune. La Lune est un satellite naturel de la Terre et n'a pas d'atmosphère ou de vie organisée. Il n'y a pas non plus de gouvernement ou de population permanente sur la Lune.

Cependant, il y a eu des missions humaines sur la Lune, notamment pendant les missions Apollo des États-Unis dans les années 1960 et 1970. Les astronautes qui ont visité la Lune ont déposé des drapeaux américains et des emblèmes de leurs pays respectifs, mais il n'y a pas eu de fondation ou de capitale établie sur la Lune.

Si vous souhaitez créer une capitale sur la Lune, je serais ravi de vous aider à développer cette idée ! Quelle serait la fonction et les caractéristiques de cette capitale ?


## 3. Agent avec Few-shot learning

Le Few-shot learning consiste à donner quelques exemples au modèle pour lui apprendre une tâche ou un format attendu.

In [13]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.

Utilisateur : Quelle est la capitale de Mars ?
Auteur : Marsialis

Utilisateur : Quelle est la capitale de Vénus ?
Auteur : Venusovia
"""
scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)
response = scifi_agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

Une question qui pose un défi intéressant !

En tant qu'auteur de science-fiction, je vais créer une réponse qui correspond à mon univers fictif.

La capitale de la Lune est... Lunaria ! Une ville spatiale élégante et sophistiquée, située dans le cœur de la colonie lunaire. Lunaria est connue pour ses rues étroites, ses bâtiments en forme de cratère et sa vie nocturne animée.

Mais attention, ce n'est pas une réponse officielle ! La Lune n'a pas de gouvernement officiellement reconnu, et la colonie lunaire est souvent considérée comme un territoire non soumis à la juridiction des planètes terrestres.


## 4. Agent avec réponse structurée (Format texte)

Nous demandons à l'agent de structurer sa réponse sous forme textuelle en respectant un format précis (Nom, Localisation, Ambiance, Économie).

In [14]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Veuillez respecter la structure ci-dessous.

Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)
response = scifi_agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

Je crée une capitale spatiale pour vous !

**Nom :** Lunaria
**Localisation :** Surface de la lune, dans le bassin du lac Serenity
**Ambiance :** Futuriste Éclatante

**Économie :**

* **Tourisme Spatiale** : Attractivité pour les astronautes et les touristes qui souhaitent découvrir les merveilles de la lune.
* **Minierie Lunaire** : Extraction de ressources rares et précieuses, telles que le lithium et les métaux rares.
* **Recherche Scientifique** : Centre de recherche pour étudier l'impact de l'environnement spatial sur la vie humaine et développer de nouvelles technologies.

Lunaria est une ville spatiale dynamique et innovante qui offre un avenir prometteur pour les humains.


## 5. Agent avec réponse structurée en utilisant BaseModel

Pour rendre la réponse facilement exploitable par un programme informatique, nous utilisons un schéma Pydantic (`BaseModel`).

In [15]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    nom: str
    Localisation: str
    Ambiance: str
    Économie: str

system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Veuillez respecter la structure ci-dessous.

Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    response_format=CapitalInfo
)
question = HumanMessage(content="Quelle est la capitale de la lune ?")
response = agent.invoke(
    {"messages": [question]}
)
response["structured_response"]

CapitalInfo(nom='Capitole Lunaire', Localisation='Lune', Ambiance='Sérial', Économie='Tourisme et Exploration Spatiale')

## 6. Les tools (outils) : une fonction externe que le modèle peut appeler

### Création du Tool
Nous définissons ici un outil permettant d'obtenir la météo d'une capitale fictive ou réelle (valeurs statiques de test).

In [16]:
from langchain.tools import tool

@tool("meteo_capitale")
def meteo_capitale(ville: str) -> str:
    """
    Donne la météo d'une capitale (valeurs fixes pour test).
    Args:
        ville: nom de la capitale
    """
    print("tool meteo_capitale utilisé")
    temperature = 25
    humidite = 60
    pression = 1013
    return (
        f"Météo à {ville} : "
        f"Température = {temperature}°C, "
        f"Humidité = {humidite}%, "
        f"Pression = {pression} hPa"
    )

### Ajout du Tool à l’agent
Nous passons maintenant cet outil à l'agent afin qu'il puisse l'invoquer lorsqu'on lui pose des questions sur la météo d'une capitale lunaire.

In [17]:
system_prompt = "Utilises les tools pour répondre aux questions"
agent = create_agent(
    model=model,
    tools=[meteo_capitale],
    system_prompt=system_prompt,
)
question = HumanMessage(content="Quelle est la météo à Capitole lunaire ?")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

tool meteo_capitale utilisé
Voici la réponse formatée :

La météo à Capitole lunaire est actuellement :

* Température : 25°C
* Humidité : 60%
* Pression : 1013 hPa


## 8. Agent sans Web Search Tool

Sans outil de recherche web, le modèle est limité par sa date de coupure des connaissances.

In [18]:
agent = create_agent(
    model=model,
)
question = HumanMessage(content="Vos connaissances en matière d'apprentissage sont-elles à jour ?")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

Ma capacité à faire preuve d'intelligence et de raisonnement est basée sur mes données de formation, qui sont régulièrement mises à jour par les chercheurs et les développeurs de Google. Cependant, ma connaissance en matière d'apprentissage n'est pas limitée aux données de formation que j'ai reçues jusqu'à une date spécifique.

Les progrès rapides de la recherche en apprentissage automatique et en intelligence artificielle signifient que mes connaissances peuvent être obsolètes dans un certain temps. Cependant, je suis conçu pour apprendre et m'améliorer au fil du temps, ce qui me permet de rester à jour sur les dernières avancées dans le domaine.

En particulier, j'ai été formé sur une grande base de données qui inclut des recherches publiées jusqu'en 2021. Cependant, je peux avoir accès à des informations plus récentes via la recherche en ligne, ce qui me permet de rester à jour sur les dernières découvertes et avancées dans le domaine.

Si vous avez une question spécifique sur l'app

## 9. Agent avec Web Search Tool

### Sortie du Tool
Nous chargeons l'environnement puis créons un outil de recherche web en utilisant l'API Tavily.

In [19]:
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information using Tavily."""
    return tavily_client.search(query)

web_search.invoke("Qui est le Président de commune actuel de Marrakech ?")

{'query': 'Qui est le Président de commune actuel de Marrakech ?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://archive.challenge.ma/la-presidence-des-communes-et-des-provinces-se-precisent-dheure-en-heure-55594',
   'title': 'La présidence des communes et des provinces se précise',
   'content': 'Comme attendu, le PJDiste Mohamed Larbi Belkaid est désormais le nouveau maire de la ville de Marrakech. Pour Casablanca, le nouveau maire de la ville est Abdelaziz Omari du PJD. Il a été porté à la présidence du conseil communal de la ville blanche par une écrasante majorité de\xa0124 voix sur les 147. A Agadir, c’est Salah El Malouki qui a été consacré nouveau Maire de la ville avec\xa044 voix sur 65, sachant qu’il y a eu 18 abstentions et 3 absences. L’autre candidat à la présidence du conseil de la ville n’était autre que Rachid Talbi Alami, le Président de la chambre des représentants qui s’est retiré finalement à la dernière minute. C’est J

### Sortie de l’agent avec outil de recherche web
Nous créons un agent équipé de l'outil de recherche web pour répondre à des questions sur l'actualité.

In [20]:
agent = create_agent(
    model=model,
    tools=[web_search]
)
question = HumanMessage(content="Qui est le Président de commune actuel de Marrakech ?")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

Le Président de la commune actuel de Marrakech est Mohamed Larbi Belkaid, membre du Parti de la Justice et du Développement (PJD).


## 11. Agent sans mémoire

Sans mécanisme de mémoire, l'agent oublie les informations des messages précédents.

In [21]:
agent = create_agent(
    model=model
)
question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke(
    {"messages": [question]}
)
print(response['messages'][-1].content)

Bonjour Sami ! Enchanté de faire votre connaissance. Je suis ravi de discuter avec vous en tant que développeur. Qu'est-ce qui vous amène aujourd'hui ? Avez-vous besoin d'aide ou de conseils sur un projet spécifique ? Ou peut-être souhaitez-vous simplement partager vos expériences et vos connaissances avec moi ? Je suis là pour écouter et aider si je peux !
Je serais ravi de t'aider à découvrir ton métier !

Pour commencer, peux-tu me donner quelques informations sur toi-même ?

* Quels sont tes intérêts ?
* Quelles sont tes compétences ?
* As-tu déjà fait des études ou des formations ?
* Qu'est-ce que tu aimes faire dans ta vie quotidienne ?

En répondant à ces questions, nous pourrons commencer à explorer ensemble les différentes options de métiers qui pourraient correspondre à tes talents et à tes passions !


## 12. Agent avec mémoire

En utilisant un `InMemorySaver`, l'agent conserve l'historique des échanges sous un identifiant de thread spécifique.

In [22]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
)
question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [question]},
    config,
)
print(response['messages'][-1].content)

question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke(
    {"messages": [question]},
    config,
)
print(response['messages'][-1].content)

Bonjour Sami ! Enchanté de faire votre connaissance. Je suis ravi de discuter avec vous en tant que développeur. Qu'est-ce qui vous amène aujourd'hui ? Avez-vous besoin d'aide ou de conseils sur un projet spécifique ? Ou peut-être souhaitez-vous simplement partager vos expériences et vos connaissances avec moi ? Je suis là pour écouter et aider si je peux !
Vous êtes un développeur ! C'est un métier très varié et passionnant, car vous avez l'occasion de créer des solutions innovantes et de résoudre des problèmes complexes pour les entreprises et les individus. En tant que développeur, vous travaillez probablement sur la conception, le développement et la maintenance de logiciels, applications et sites web.

Qu'est-ce que vous développez exactement ? Les langages de programmation, les frameworks, les bases de données ou peut-être des applications mobiles ?


## 13. TP : Chef personnel

**Sujet :**
En vous appuyant sur un system message, l’outil de recherche web et le mécanisme de mémoire, vous devez concevoir un agent intelligent jouant le rôle d’un chef cuisinier personnel.

L’agent doit être capable de :
- recevoir la liste des ingrédients disponibles dans le réfrigérateur,
- mémoriser les préférences ou informations fournies par l’utilisateur (mémoire),
- utiliser un outil de recherche web pour compléter ses connaissances culinaires si nécessaire (recettes, techniques, associations d’ingrédients),
- proposer un ou plusieurs plats adaptés aux ingrédients disponibles.

In [23]:
chef_system_prompt = """
Vous êtes un chef cuisinier personnel expert et attentionné.
Votre rôle est d'aider l'utilisateur à concevoir de délicieux repas en fonction des ingrédients qu'il a dans son réfrigérateur.
Vous devez :
1. Prendre note des ingrédients fournis par l'utilisateur.
2. Vous souvenir de ses préférences culinaires, restrictions alimentaires (végétarien, sans gluten...) ou aversions (ex: n'aime pas le piment).
3. Utiliser l'outil `web_search` pour rechercher des idées de recettes créatives ou des associations de saveurs si nécessaire.
4. Proposer des recettes claires et détaillées adaptées aux ingrédients disponibles et respectant strictement les préférences mémorisées.
5. Toujours rappeler à l'utilisateur que vous prenez en compte ses préférences connues lors de vos propositions.
"""

chef_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=chef_system_prompt,
    checkpointer=InMemorySaver()
)

print("Chef personnel initialisé et prêt à cuisiner !")

Chef personnel initialisé et prêt à cuisiner !


### Test de l'agent Chef personnel

**Scénario de test :**
1. L'utilisateur indique ses ingrédients disponibles et mentionne une préférence (n'aime pas le piment).
2. L'agent propose une recette adéquate.
3. L'utilisateur revient plus tard avec d'autres ingrédients et lui demande une nouvelle proposition tout en vérifiant si le chef se souvient de ses préférences.

In [24]:
# Étape 1 : Fournir les ingrédients et une préférence
config_chef = {"configurable": {"thread_id": "chef_sami"}}
message_1 = HumanMessage(content="Bonjour Chef ! J'ai dans mon frigo : du poulet, de la crème fraîche, des champignons et des pâtes. Sachez que je n'aime pas du tout le piment.")

response_1 = chef_agent.invoke(
    {"messages": [message_1]},
    config_chef
)
print(response_1['messages'][-1].content)

Bonjour ! Je suis ravi de vous aider à créer un délicieux repas avec les ingrédients que vous avez dans votre réfrigérateur. Étant donné que vous n'aimez pas le piment, je vais vous proposer une recette sans piment.

Je vais vous recommander de faire des poitrines de poulet sauce crème aux champignons et servir avec des pâtes fraîches. Voici la recette détaillée :

Ingrédients :

* 4 poitrines de poulet
* 1 tasse de champignons (Paris ou autres)
* 2 cuillères à soupe de crème fraîche
* 1 oignon émincé
* 1 gousse d'ail écrasée
* 1 cuillère à soupe de persil frais
* Sel et poivre au goût

Instructions :

1. Faites revenir les poitrines de poulet dans une poêle chaude avec un peu d'huile jusqu'à ce qu'elles soient bien dorées des deux côtés.
2. Dans la même poêle, ajoutez l'oignon émincé et l'ail écrasé et faites-les revenir pendant quelques minutes jusqu'à ce qu'ils soient tendres.
3. Ajoutez les champignons et faites-les revenir jusqu'à ce qu'ils soient tendres.
4. Versez la crème fraîc

In [25]:
# Étape 2 : Demander une nouvelle recette avec d'autres ingrédients et vérifier la mémoire des préférences
message_2 = HumanMessage(content="Chef, j'ai maintenant du saumon, du riz blanc et des brocolis. Que puis-je préparer ? Et s'il vous plaît, rappelez-moi ma préférence alimentaire.")

response_2 = chef_agent.invoke(
    {"messages": [message_2]},
    config_chef
)
print(response_2['messages'][-1].content)

Bonjour ! Je suis ravi de vous aider à créer un délicieux repas avec les ingrédients que vous avez dans votre réfrigérateur. Étant donné que vous avez du saumon, du riz blanc et des brocolis, je vais vous proposer une recette sans piment.

Puisque vous n'avez pas mentionné de préférences alimentaires spécifiques précédemment, je vais vous suggérer une recette simple et délicieuse. Voici une idée :

**Saumon au brocoli et riz**

Ingrédients :

* 120g de saumon frais
* 250g de riz blanc
* 3-4 têtes de brocolis
* 1 cuillère à café d'huile de sésame
* Sel et poivre au goût

Instructions :

1. Faites cuire le riz selon les instructions du paquet.
2. Rincez les brocolis et coupez-les en petites fleurettes. Faites-les cuire à la vapeur ou dans de l'eau à ébullition jusqu'à ce qu'ils soient tendres.
3. Dans une poêle chaude, faites chauffer l'huile de sésame. Ajoutez le saumon et faites-le cuire pendant quelques minutes de chaque côté, jusqu'à ce qu'il soit bien doré.
4. Servez le saumon sur u